# 06 — LoRA fine-tuning as a controlled change

        **Estimated time:** 55 minutes<br>
        **Prerequisites:** 05 — Prompt baselines<br>
        **Learner-produced evidence:** a reviewed training configuration and measured adapter evidence

        ## Learning objectives

        - Explain which weights LoRA changes and which base artifacts remain fixed.
- Connect the portable chat records to the MLX-LM training configuration.
- Interpret loss and memory as training evidence, not adoption evidence.

        This notebook is a teaching interface over the reusable code in `src/`.
        It uses only prepared local files. Run `make prepare-flight` before the
        trip; no cell installs packages or downloads data.


In [ ]:
from aai_local_finetuning.offline import enable_offline_environment

enable_offline_environment()

## Inspect the change before running it

The base model revision and 4-bit weights stay fixed. LoRA learns a small
adapter over selected projections. The configuration is versioned so the
change can be reproduced and hashed with its later evaluation evidence.


In [ ]:
import json

import yaml

from aai_local_finetuning.settings import PROJECT_ROOT, load_settings

settings = load_settings()
config_path = PROJECT_ROOT / "configs" / "training" / "lora.yaml"
training_config = yaml.safe_load(config_path.read_text(encoding="utf-8"))
training_config

## Read one portable training example

MLX-LM consumes the same framework-neutral messages a future trainer can
consume. Metadata is evidence and slicing context; it is not appended to
the user prompt as a shortcut to the target.


In [ ]:
first_training_record = json.loads(
    (settings.processed_dir / "train.jsonl").read_text(encoding="utf-8").splitlines()[0]
)
{
    "example_id": first_training_record["example_id"],
    "message_roles": [message["role"] for message in first_training_record["messages"]],
    "user_preview": first_training_record["messages"][1]["content"][:140],
    "assistant_target": json.loads(first_training_record["messages"][2]["content"]),
    "metadata_keys": sorted(first_training_record["metadata"]),
}

## What the conservative settings buy us

Batch size 1, gradient accumulation, eight adapted layers, a bounded
sequence length, prompt masking, and checkpointing reduce unified-memory
pressure. They are design choices for the prepared 24 GB Apple-silicon
machine, not universal optimal values.


In [ ]:
training_anatomy = {
    key: training_config.get(key)
    for key in (
        "model",
        "train",
        "fine_tune_type",
        "num_layers",
        "batch_size",
        "grad_accumulation_steps",
        "max_seq_length",
        "mask_prompt",
        "grad_checkpoint",
        "iters",
        "seed",
    )
}
training_anatomy["lora_parameters"] = training_config.get("lora_parameters")
training_anatomy

## Load measured preflight evidence

Flight preparation already runs one real iteration to catch MLX compile,
data-shape, and memory surprises. This is a readiness probe, not the final
change. A missing file tells you preparation did not complete.


In [ ]:
preflight_path = PROJECT_ROOT / "artifacts" / "training" / "preflight-smoke.json"
preflight_evidence = (
    json.loads(preflight_path.read_text(encoding="utf-8"))
    if preflight_path.is_file()
    else {"status": "missing; prepare this machine online"}
)
preflight_evidence

## Optional live training cell

The default is safe for Run All. Set `RUN_TRAINING = True` to run ten
iterations into a notebook-specific adapter directory. This never
overwrites the canonical `bitext-lora-v1` change. For the full configured
run, set `TRAINING_ITERATIONS = None` only after the smoke evidence looks
healthy and you have enough time and battery.


In [ ]:
from aai_local_finetuning.training import run_lora

RUN_TRAINING = False
TRAINING_ITERATIONS = 10
notebook_adapter = PROJECT_ROOT / "artifacts" / "notebook" / "adapters" / "bitext-smoke"
if RUN_TRAINING:
    training_evidence = run_lora(
        iterations=TRAINING_ITERATIONS,
        adapter_path=notebook_adapter,
        log_name="notebook-bitext-smoke",
    ).model_dump(mode="json")
else:
    training_evidence = {
        "status": "skipped",
        "how_to_run": "Set RUN_TRAINING = True",
        "preflight": preflight_evidence,
    }
training_evidence

## Exercise — interpret optimization evidence

After a run, compare training and validation losses and measured peak
memory. Success means your conclusion avoids claiming that lower loss
proves better frozen-test behavior or safer responses.


In [ ]:
optimization_conclusion = (
    "The adapter optimization executed locally within the measured memory "
    "profile. Only the frozen structured-output evaluation can determine "
    "whether the change should be adopted."
)
assert "frozen" in optimization_conclusion.lower()
optimization_conclusion

**Hint:** loss is calculated on the training objective. Promotion asks a
broader question about generalization, schema, labels, policy, latency,
tokens, memory, and the strongest meaningful baseline.


## Checkpoint

You can now name the adapter as the change and explain why a successful
training process is necessary but insufficient evidence.

**Next:** `07_frozen_evaluation.ipynb` opens the frozen test once and
applies the already locked methods.
